In [3]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta
from collections import defaultdict


In [6]:
# ============================================================
# UPDATE THIS PATH to your transaction file
# ============================================================
INPUT_FILE = "../outputs_updated/transactions_generated_typology_V2.parquet"

# ── Fix pyarrow extension type conflicts (Python 3.13 + pandas bug) ──
try:
    import pyarrow as pa
    for ext_name in ["pandas.period", "pandas.interval", "arrow.py_extension_type"]:
        try:
            pa.unregister_extension_type(ext_name)
        except Exception:
            pass
except ImportError:
    pass

# Bank column name -> internal clean name
COLUMN_MAP = {
    "Transaction ID/Reference No":               "txn_id",
    "Timestamp":                                  "timestamp",
    "Datestamp":                                   "datestamp",
    "Transaction Amount":                          "amount",
    "Currency":                                    "currency",
    "Transaction Type":                            "txn_type",
    "Transaction Mode/Channel - Bank":             "channel_bank",
    "Cash Flag":                                   "cash_flag",
    "Transaction Mode/Channel - PPI":              "channel_ppi",
    "Transaction Status":                          "txn_status",
    "Wallet Balance Before":                       "wallet_bal_before",
    "Wallet Balance After":                        "wallet_bal_after",
    "Source of Funds - Wallet":                    "source_funds_wallet",
    "Load Instrument Type":                        "load_instrument",
    "Load Source Account/Card Details":            "load_source_masked",
    "Beneficiary Wallet ID/VPA for UPI":           "beneficiary_vpa",
    "Merchant ID":                                 "merchant_id",
    "Merchant Name":                               "merchant_name",
    "Merchant Category Code (MCC)":                "mcc",
    "Merchant Location":                           "merchant_location",
    "Refund/Chargeback Flag":                      "refund_flag",
    "Customer Account Number":                     "account_number",
    "Account/Wallet Status":                       "account_status",
    "Non Face to Face Flag":                       "non_f2f_flag",
    "PEP Flag":                                    "pep_flag",
    "HNI Flag":                                    "hni_flag",
    "Minor Flag":                                  "minor_flag",
    "Customer Branch IFSC Code":                   "branch_ifsc",
    "Customer CIF/ID Number":                      "cif_id",
    "Customer CIF/ID Number Creation Date":        "cif_creation_date",
    "Annual Income":                               "annual_income",
    "Counterparty Account Number":                 "cp_account_number",
    "Counterparty Branch IFSC/Swift Code":         "cp_ifsc_swift",
    "Customer Name":                               "customer_name",
    "Counterparty Name":                           "cp_name",
    "Sender Country Code*":                        "sender_country",
    "Receiver Country Code*":                      "receiver_country",
    "Customer Current Risk Score":                 "risk_score",
    "Customer Type":                               "customer_type",
    "Customer Entity Type":                        "entity_type",
    "Account Category":                            "account_category",
    "Account Type":                                "account_type",
    "Account/Wallet Opening Date":                 "account_open_date",
    "Customer Occupation/Industry":                "occupation",
    "VKYC Flag":                                   "vkyc_flag",
    "KYC Update Date":                             "kyc_update_date",
    "Account/Wallet Inoperative Status Date":      "inoperative_date",
    "Source of Funds":                              "source_of_funds",
    "Tax Residency":                               "tax_residency",
    "Nationality":                                 "nationality",
    "Citizenship":                                 "citizenship",
    "Residency":                                   "residency",
    "Date of Incorporation/Formation":             "incorporation_date",
    "Place of Incorporation/Formation":            "incorporation_place",
    "Beneficial Owner Types":                      "bo_types",
    "Passive NFE":                                 "passive_nfe",
    "Address of Registered Office":                "addr_registered",
    "Address of Place of Business":                "addr_business",
    "Address of Beneficial Owners/Related Persons":"addr_bo",
    "Address of Individual Customer":              "addr_individual",
    "Date of Birth":                               "dob",
    "Father/Spouse Name":                          "father_spouse",
    "Identification Proof Doc No":                 "id_doc_no",
    "Entity Identification Proof Doc No":          "entity_id_doc_no",
    "Credit Summation of the account for the period": "credit_sum_period",
    "Debit Summation of the account for the period":  "debit_sum_period",
    "Professional Experience in Years - Individual":   "experience_years",
    "CIF/ID of Beneficial Owners/Related Persons":     "cif_bo",
    "Name of Beneficial Owners/Related Persons":       "name_bo",
    "Mobile Number":                               "mobile",
    "PAN":                                         "pan",
    "Aadhaar Number":                              "aadhaar",
    "Email ID":                                    "email",
    "Wallet KYC Category":                         "wallet_kyc",
    "Wallet Account ID":                           "wallet_id",
    "Escrow Account Linked":                       "escrow_account",
    "Transaction Limit (Per Transaction)":         "limit_per_txn",
    "Daily Transaction Limit":                     "limit_daily",
    "Monthly Transaction Limit":                   "limit_monthly",
    "Annual Transaction Limit":                    "limit_annual",
    "Maximum Wallet Balance Limit":                "max_wallet_bal",
    "Device ID/Fingerprint":                       "device_id",
    "IP Address of Originating Device":            "ip_address",
    "Geo-Location (City/Country)":                 "geo_location",
    "GPS Coordinates":                             "gps_coords",
    "Browser/App Information":                     "browser_app",
    "Session ID":                                  "session_id",
    "Authentication Method (OTP/PIN/Biometric)":   "auth_method",
    "VPN Flag":                                    "vpn_flag",
    "Emulator Flag":                               "emulator_flag",
    "Lat/Long of Customer Address":                "customer_latlon",
}

def load_transactions(filepath):
    ext = filepath.rsplit('.', 1)[-1].lower()
    if ext == 'parquet':
        try:
            import pyarrow.parquet as pq
            table = pq.read_table(filepath)
            raw = table.to_pandas()
        except Exception as e:
            print(f"  Parquet read failed ({e}), trying CSV fallback...")
            csv_path = filepath.replace('.parquet', '.csv')
            if os.path.exists(csv_path):
                raw = pd.read_csv(csv_path, low_memory=False)
            else:
                raise
    elif ext in ('xlsx', 'xls'):
        raw = pd.read_excel(filepath)
    else:
        raw = pd.read_csv(filepath, low_memory=False)
    print(f"Raw data loaded: {len(raw):,} rows x {len(raw.columns)} columns")

    # Handle duplicate "Transaction Type" columns
    cols = list(raw.columns)
    seen_txn_type = False
    for i, c in enumerate(cols):
        if c.strip() == "Transaction Type":
            if seen_txn_type:
                cols[i] = "Transaction Type PPI"
                COLUMN_MAP["Transaction Type PPI"] = "txn_type_ppi"
            seen_txn_type = True
    raw.columns = cols

    rename = {}
    for orig, clean in COLUMN_MAP.items():
        for col in raw.columns:
            if col.strip() == orig.strip():
                rename[col] = clean
                break
    raw = raw.rename(columns=rename)
    return raw

# Load — try parquet first, then CSV, then alternatives
if os.path.exists(INPUT_FILE):
    df = load_transactions(INPUT_FILE)
elif os.path.exists(INPUT_FILE.replace('.parquet', '.csv')):
    df = load_transactions(INPUT_FILE.replace('.parquet', '.csv'))
else:
    print(f"'{INPUT_FILE}' not found. Looking for alternatives...")
    found = False
    for alt in ["../outputs_updated/transactions_generated_typology_V2.csv",
                "../outputs_updated/transactions_generated_typology_V2.parquet",
                "../outputs_updated/transactions_generated_typology_V2.csv",
                "../transactions_generated_typology_V2.csv",
                "../transactions_generated_typology_V2.parquet"]:
        if os.path.exists(alt):
            df = load_transactions(alt)
            found = True
            break
    if not found:
        raise FileNotFoundError("No transaction file found. Export CSV from creator notebook.")

Raw data loaded: 333,875 rows x 97 columns


In [7]:
df.columns

Index(['transaction_id', 'timestamp', 'datestamp', 'transaction_amount',
       'currency', 'transaction_type_dr_cr', 'transaction_mode_channel_bank',
       'cash_flag', 'transaction_type_ppi', 'transaction_mode_channel_ppi',
       'transaction_status', 'wallet_balance_before', 'wallet_balance_after',
       'source_of_funds_wallet', 'load_instrument_type',
       'load_source_account_card_details', 'beneficiary_wallet_id_vpa',
       'merchant_id', 'merchant_name', 'merchant_category_code',
       'merchant_location', 'refund_chargeback_flag',
       'customer_account_number', 'account_wallet_status',
       'non_face_to_face_flag', 'pep_flag', 'hni_flag', 'minor_flag',
       'customer_branch_ifsc_code', 'customer_cif_id',
       'customer_cif_creation_date', 'annual_income',
       'counterparty_account_number', 'counterparty_branch_ifsc_swift',
       'customer_name', 'counterparty_name', 'sender_country_code',
       'receiver_country_code', 'customer_current_risk_score', 'custo

In [13]:
df.columns = df.columns.str.strip()

# Normalize key fields
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["transaction_amount"] = df["transaction_amount"].astype(float)

# Direction helpers
df["is_credit"] = df["transaction_type_dr_cr"].str.lower().str.contains("credit")
df["is_debit"]  = df["transaction_type_dr_cr"].str.lower().str.contains("debit")


In [14]:


# ============================================================
# STORAGE
# ============================================================
flags = []
scenarios = []

def make_scenario_id(accounts, typology):
    return f"{typology}_" + str(hash(frozenset(accounts)))

# ============================================================
# 1. STRUCTURING (STRICT)
# ============================================================
def detect_structuring(df):
    typ = "Structuring"
    grouped = df[df["channel_bank"] == "Branch Cash"]

    for acc, g in grouped.groupby("account_number"):
        g = g.sort_values("timestamp")

        deposits = g[
            (g["is_credit"]) &
            (g["amount"].between(8000, 9900))
        ]

        if len(deposits) < 3:
            continue

        window_start = deposits["timestamp"].min()
        window_end = window_start + timedelta(days=3)

        deposits = deposits[deposits["timestamp"] <= window_end]

        if deposits["account_number"].nunique() < 3:
            continue

        # find consolidation
        transfers = df[
            (df["account_number"].isin(deposits["account_number"])) &
            (df["is_debit"]) &
            (df["amount"].between(7500, 9800)) &
            (df["timestamp"].between(window_start, window_start + timedelta(days=7)))
        ]

        if len(transfers) == 0:
            continue

        accounts = set(deposits["account_number"])
        sid = make_scenario_id(accounts, typ)

        scenarios.append((sid, typ, accounts))
        flags.extend(deposits["txn_id"].tolist())
        flags.extend(transfers["txn_id"].tolist())


# ============================================================
# 2. FUNNEL (STRICT)
# ============================================================
def detect_funnel(df):
    typ = "Funnel"

    for acc, g in df.groupby("account_number"):
        inflow = g[g["is_credit"]]

        senders = inflow["cp_account_number"].dropna().unique()
        if len(senders) < 15:
            continue

        total = inflow["amount"].sum()
        if total < 75000:
            continue

        outflow = g[g["is_debit"]]

        if len(outflow) < 2:
            continue

        forwarded = outflow["amount"].sum()
        retention = 1 - (forwarded / total)

        if not (0.03 <= retention <= 0.07):
            continue

        accounts = set([acc]) | set(senders)
        sid = make_scenario_id(accounts, typ)

        scenarios.append((sid, typ, accounts))
        flags.extend(g["txn_id"].tolist())


# ============================================================
# 3. PASS THROUGH
# ============================================================
def detect_passthrough(df):
    typ = "PassThrough"

    for acc, g in df.groupby("account_number"):
        g = g.sort_values("timestamp")

        for i in range(len(g)-1):
            a = g.iloc[i]
            b = g.iloc[i+1]

            if not (a["is_credit"] and b["is_debit"]):
                continue

            if not (200000 <= a["amount"] <= 2000000):
                continue

            gap = (b["timestamp"] - a["timestamp"]).total_seconds() / 3600
            if gap > 1:
                continue

            pct = b["amount"] / a["amount"]
            if not (0.96 <= pct <= 0.99):
                continue

            accounts = {acc, a["cp_account_number"], b["cp_account_number"]}
            sid = make_scenario_id(accounts, typ)

            scenarios.append((sid, typ, accounts))
            flags.extend([a["txn_id"], b["txn_id"]])


# ============================================================
# 4. LAYERING (STRICT CHAIN)
# ============================================================
def detect_layering(df):
    typ = "Layering"

    df_sorted = df.sort_values("timestamp")

    for i in range(len(df_sorted)):
        chain = [df_sorted.iloc[i]]
        current = df_sorted.iloc[i]

        for j in range(i+1, len(df_sorted)):
            nxt = df_sorted.iloc[j]

            if nxt["account_number"] != current["cp_account_number"]:
                continue

            time_gap = (nxt["timestamp"] - current["timestamp"]).total_seconds()/60
            if not (5 <= time_gap <= 30):
                break

            ratio = nxt["amount"] / current["amount"]
            if not (0.97 <= ratio <= 1.0):
                break

            chain.append(nxt)
            current = nxt

            if len(chain) >= 8:
                accounts = set(x["account_number"] for x in chain)
                sid = make_scenario_id(accounts, typ)

                scenarios.append((sid, typ, accounts))
                flags.extend([x["txn_id"] for x in chain])
                break


# ============================================================
# 5. THIRD PARTY WEB
# ============================================================
def detect_third_party(df):
    typ = "ThirdParty"

    for acc, g in df.groupby("account_number"):
        inflow = g[g["is_credit"]]

        payers = inflow["cp_account_number"].dropna().unique()
        if len(payers) < 5:
            continue

        if not inflow["amount"].between(10000, 100000).all():
            continue

        accounts = set([acc]) | set(payers)
        sid = make_scenario_id(accounts, typ)

        scenarios.append((sid, typ, accounts))
        flags.extend(inflow["txn_id"].tolist())


# ============================================================
# 6. MONEY MULE
# ============================================================
def detect_mule(df):
    typ = "Mule"

    for acc, g in df.groupby("account_number"):
        out = g[g["is_debit"]]

        if len(out) < 5:
            continue

        ratios = out["amount"] / g["amount"].max()
        if not ((ratios > 0.85) & (ratios < 0.95)).all():
            continue

        accounts = set([acc]) | set(out["cp_account_number"])
        sid = make_scenario_id(accounts, typ)

        scenarios.append((sid, typ, accounts))
        flags.extend(g["txn_id"].tolist())


# ============================================================
# 7. HIGH RISK CORRIDOR
# ============================================================
def detect_corridor(df):
    typ = "Corridor"

    risky = ["AE","PK","BD","NP","LK","MM","AF"]

    g = df[df["receiver_country"].isin(risky)]

    for acc, grp in g.groupby("account_number"):
        if len(grp) < 3:
            continue

        if not grp["amount"].between(50000, 500000).all():
            continue

        accounts = set([acc]) | set(grp["cp_account_number"])
        sid = make_scenario_id(accounts, typ)

        scenarios.append((sid, typ, accounts))
        flags.extend(grp["txn_id"].tolist())


# ============================================================



In [15]:
detect_structuring(df)

KeyError: 'channel_bank'

In [ ]:
# RUN ALL
# ============================================================
detect_structuring(df)
detect_funnel(df)
detect_passthrough(df)
detect_layering(df)
detect_third_party(df)
detect_mule(df)
detect_corridor(df)

# ============================================================
# OUTPUT
# ============================================================
df["flagged"] = df["txn_id"].isin(set(flags))

df_flags = df[df["flagged"]]
df_scenarios = pd.DataFrame(scenarios, columns=["scenario_id","typology","accounts"])

print("\n=== RESULTS ===")
print("Flagged txns:", len(df_flags))
print("Scenarios:", len(df_scenarios))

print("\nBy typology:")
print(df_scenarios["typology"].value_counts())